# Tratamento inicial dos dados

Este notebook parte do diagnóstico realizado em `01_entendimento_dos_dados.ipynb` e cria uma base preparada para análise. Os arquivos brutos não serão alterados.

## Regras adotadas

- `constant column`: remove colunas sem variação, pois elas não ajudam a diferenciar observações.
- `missing` acima de 50%: remove a coluna, pois existe pouca informação observada para representá-la com segurança.
- `missing` restante: preenche com a mediana da própria coluna. A mediana sofre menos influência de `outliers` (valores extremos) que a média.

Essas regras são uma preparação inicial para análise exploratória. Elas poderão ser reavaliadas durante a modelagem e a validação dos resultados.

In [1]:
from pathlib import Path
import re

import pandas as pd

raiz_projeto = Path.cwd().parent
pasta_dados = raiz_projeto / "data" / "raw"
caminho_dados = pasta_dados / "secom.data"
caminho_rotulos = pasta_dados / "secom_labels.data"

dados = pd.read_csv(
    caminho_dados,
    sep=r"\s+",
    header=None,
    na_values="NaN",
    engine="python",
)

padrao_rotulo = re.compile(r'^\s*(-?\d+)\s+"([^"]+)"\s*$')
linhas_rotulos = caminho_rotulos.read_text(encoding="utf-8").splitlines()
rotulos_extraidos = [
    padrao_rotulo.match(linha).groups()
    for linha in linhas_rotulos
    if padrao_rotulo.match(linha)
]
rotulos = pd.DataFrame(rotulos_extraidos, columns=["rotulo", "timestamp"])
rotulos["rotulo"] = rotulos["rotulo"].astype(int)
rotulos["timestamp"] = pd.to_datetime(
    rotulos["timestamp"],
    format="%d/%m/%Y %H:%M:%S",
)
rotulos["falha"] = (rotulos["rotulo"] == 1).astype(int)

print(f"Dados carregados: {dados.shape}")
print(f"Rótulos carregados: {rotulos.shape}")

Dados carregados: (1567, 590)
Rótulos carregados: (1567, 3)


In [2]:
# Mantém a leitura original intacta e trabalha em uma cópia.
dados_tratados = dados.copy()

percentual_missing = dados_tratados.isna().mean()
colunas_constantes = dados_tratados.columns[
    dados_tratados.nunique(dropna=True) <= 1
]
colunas_excesso_missing = percentual_missing[
    percentual_missing > 0.50
].index
colunas_removidas = colunas_constantes.union(colunas_excesso_missing)

dados_tratados = dados_tratados.drop(columns=colunas_removidas)

medianas = dados_tratados.median(numeric_only=True)
dados_tratados = dados_tratados.fillna(medianas)

base_modelagem = dados_tratados.copy()
base_modelagem["falha"] = rotulos["falha"].to_numpy()

resumo_tratamento = pd.DataFrame(
    {
        "métrica": [
            "features antes do tratamento",
            "features removidas",
            "features depois do tratamento",
            "missing antes do tratamento",
            "missing depois do tratamento",
            "linhas da base de modelagem",
        ],
        "valor": [
            int(dados.shape[1]),
            int(len(colunas_removidas)),
            int(dados_tratados.shape[1]),
            int(dados.isna().sum().sum()),
            int(dados_tratados.isna().sum().sum()),
            int(base_modelagem.shape[0]),
        ],
    }
)
resumo_tratamento

,métrica,valor
0,features antes do tratamento,590
1,features removidas,144
2,features depois do tratamento,446
3,missing antes do tratamento,41951
4,missing depois do tratamento,0
5,linhas da base de modelagem,1567


In [3]:
# Valida as condições mínimas para prosseguir para a análise exploratória.
assert len(dados) == len(rotulos)
assert dados_tratados.isna().sum().sum() == 0
assert base_modelagem.shape[0] == len(rotulos)
assert base_modelagem["falha"].isin([0, 1]).all()

print("Validação concluída: a base tratada não possui missing e está alinhada ao label.")
base_modelagem.head()

Validação concluída: a base tratada não possui missing e está alinhada ao label.


,0,1,2,3,4,6,7,8,9,10,...,577,582,583,584,585,586,587,588,589,falha
0,3030.93,2564.00,2187.7333,1411.1265,1.3602,97.6133,0.1242,1.5005,0.0162,-0.0034,...,14.9509,0.5005,0.0118,0.0035,2.3630,0.0205,0.0148,0.0046,71.9005,0
1,3095.78,2465.14,2230.4222,1463.6606,0.8294,102.3433,0.1247,1.4966,-0.0005,-0.0148,...,10.9003,0.5019,0.0223,0.0055,4.4447,0.0096,0.0201,0.0060,208.2045,0
2,2932.61,2559.94,2186.4111,1698.0172,1.5102,95.4878,0.1241,1.4436,0.0041,0.0013,...,9.2721,0.4958,0.0157,0.0039,3.1745,0.0584,0.0484,0.0148,82.8602,1
3,2988.72,2479.90,2199.0333,909.7926,1.3204,104.2367,0.1217,1.4882,-0.0124,-0.0033,...,8.5831,0.4990,0.0103,0.0025,2.0544,0.0202,0.0149,0.0044,73.8432,0
4,3032.24,2502.87,2233.3667,1326.5200,1.5334,100.3967,0.1235,1.5031,-0.0031,-0.0072,...,10.9698,0.4800,0.4766,0.1045,99.3032,0.0202,0.0149,0.0044,73.8432,0
